# SaluNet — Evaluate CIFAR checkpoints on Google Colab

**No local installation or files from your server are required.**
This notebook clones the repository, installs its dependencies, downloads
the Hugging Face checkpoints, and evaluates the 10,000 test images of the
corresponding dataset.

1. In Colab, select **File → Upload notebook** and choose this `.ipynb` file.
2. Select **Runtime → Change runtime type → GPU**.
3. Choose a model in the first code cell (`all` evaluates all four checkpoints).
4. Run the cells in order, or select **Runtime → Run all**.

| Identifier | Model | Dataset | Activation | EMA Top-1 (%) | EMA Top-5 (%) |
|---|---|---|---|---:|---:|
| c18_c10_galu | SaluNet-C-18 | CIFAR-10 | GALU | 97.27 | 99.89 |
| c18_c100_swalu | SaluNet-C-18 | CIFAR-100 | SWALU | 83.42 | 96.25 |
| t_c100_gelu | SaluNet-T | CIFAR-100 | GELU | 70.33 | 90.09 |
| t_c100_galu | SaluNet-T | CIFAR-100 | GALU | 70.36 | 89.08 |

Evaluation uses the architectures and `scripts/eval.py` from the
[SaluNet repository](https://github.com/MouradZaied/SaluNet), strict weight
loading, **FP32 without autocast**, and each model's evaluation transforms.
No training is performed. Reference scores correspond to the released files,
not averages across multiple training runs.

Installed packages, checkpoints, and datasets are temporary within the Colab
session. The final cell downloads the results and logs.


## 1. Select checkpoints

Select `all` to evaluate all four files. To start with a single model,
choose, for example, `c18_c100_swalu`. No Hugging Face token is required.


In [ ]:
#@title Select the model to evaluate
MODEL_CHOICE = "all" #@param ["all", "c18_c10_galu", "c18_c100_swalu", "t_c100_gelu", "t_c100_galu"]
BATCH_SIZE = 512 #@param {type:"integer"}
NUM_WORKERS = 2 #@param {type:"integer"}
DEVICE = "cuda" #@param ["cuda", "cpu"]

from pathlib import Path
import sys
import platform

REPO_DIR = Path('/content/SaluNet')
CHECKPOINT_DIR = Path('/content/salunet_checkpoints')
RESULTS_DIR = Path('/content/salunet_cifar_results')
SELECTED = (['c18_c10_galu', 'c18_c100_swalu', 't_c100_gelu', 't_c100_galu']
            if MODEL_CHOICE == 'all' else [MODEL_CHOICE])
LOCAL_CHECKPOINTS = {}
TOLERANCE_PP = 0.02

if BATCH_SIZE < 1 or NUM_WORKERS < 0:
    raise ValueError('BATCH_SIZE must be positive and NUM_WORKERS nonnegative.')
print('Selected:', SELECTED)
print('Python:', sys.executable)


## 2. Automatic setup

This cell uses Colab's preinstalled PyTorch and torchvision and preserves
their versions when installing SaluNet. It then checks imports and a GPU
computation. Initial setup may take a few minutes.

If CUDA is unavailable, enable a GPU in the Colab runtime settings and rerun
the notebook. CPU evaluation is also supported but is substantially slower.


In [ ]:
import subprocess
import importlib.metadata as metadata

# Import the Colab-provided pair before installing any project dependency.
import torch
import torchvision

if DEVICE == 'cuda':
    if not torch.cuda.is_available():
        raise RuntimeError('Enable a GPU under Runtime → Change runtime type, then rerun.')
    x = torch.randn(32, 32, device='cuda')
    y = x @ x
    torch.cuda.synchronize()
    print('GPU:', torch.cuda.get_device_name(0), flush=True)
    del x, y
    torch.cuda.empty_cache()

if not (REPO_DIR / 'scripts/eval.py').is_file():
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        raise RuntimeError(f'Existing nonempty folder is not a SaluNet clone: {REPO_DIR}')
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', 'https://github.com/MouradZaied/SaluNet.git',
                    str(REPO_DIR)], check=True)

# Preserve Colab's installed torch/torchvision, including their CUDA builds.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--no-cache-dir',
    '-e', str(REPO_DIR), f'torch=={torch.__version__}',
    f'torchvision=={torchvision.__version__}',
], check=True)

# Use a fresh process for project imports after dependency installation.
check = subprocess.run([sys.executable, str(REPO_DIR / 'scripts/eval.py'), '--help'],
                       cwd=REPO_DIR, capture_output=True, text=True)
if check.returncode:
    print(check.stdout, check.stderr)
    raise RuntimeError('Installation/import check failed. See the error above.')

VERSIONS = {'python': platform.python_version()}
for package in ('torch', 'torchvision', 'pytorch-lightning', 'torchmetrics', 'numpy', 'Pillow', 'PyYAML'):
    VERSIONS[package] = metadata.version(package)
    print(f'{package}: {VERSIONS[package]}')

REPO_REVISION = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
REPO_DIRTY = bool(subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip())
print('Repository commit:', REPO_REVISION)
print('Evaluation entry point: OK')


## 3. Checkpoints and matching configurations

Downloads come from the official Hugging Face repository. Checkpoints remain
cached in the session; their SHA-256 hashes are recorded with the results.
The repository's data pipeline downloads datasets if needed, using the YAML
`data_dir` setting (normally `./data`, relative to the repository).


In [ ]:
MODELS = {
    'c18_c10_galu': {
        'model': 'SaluNet-C-18', 'dataset': 'CIFAR-10', 'activation': 'galu',
        'config': 'configs/experiments/salunet-c-18/cifar10/salunet-c-18_cifar10.yaml',
        'filename': 'best_ema_model_SaluNet-C-18_C10_97.27.pth',
        'expected_top1': 97.27, 'expected_top5': 99.89,
    },
    'c18_c100_swalu': {
        'model': 'SaluNet-C-18', 'dataset': 'CIFAR-100', 'activation': 'swalu',
        'config': 'configs/experiments/salunet-c-18/cifar100/salunet-c-18_cifar100_deterministic.yaml',
        'filename': 'best_ema_model_SaluNet-C-18_C100_83.42.pth',
        'expected_top1': 83.42, 'expected_top5': 96.25,
    },
    't_c100_gelu': {
        'model': 'SaluNet-T', 'dataset': 'CIFAR-100', 'activation': 'gelu',
        'config': 'configs/experiments/salunet-t/cifar100/salunet-t_cifar100_salu_gelu.yaml',
        'filename': 'best_ema_model_SaluNet-T_C100_SALU_GELU_70.33.pth',
        'expected_top1': 70.33, 'expected_top5': 90.09,
    },
    't_c100_galu': {
        'model': 'SaluNet-T', 'dataset': 'CIFAR-100', 'activation': 'galu',
        'config': 'configs/experiments/salunet-t/cifar100/salunet-t_cifar100_salu_galu.yaml',
        'filename': 'best_ema_model_SaluNet-T_C100_SALU_GALU_70.36.pth',
        'expected_top1': 70.36, 'expected_top5': 89.08,
    },
}

if not SELECTED or len(set(SELECTED)) != len(SELECTED):
    raise ValueError('Select at least one model, without duplicate identifiers.')
for key in SELECTED:
    if key not in MODELS:
        raise KeyError(f'Unknown model: {key}; choices: {list(MODELS)}')
    if not (REPO_DIR / MODELS[key]['config']).is_file():
        raise FileNotFoundError(REPO_DIR / MODELS[key]['config'])
print('Selected configurations: OK')


In [ ]:
import hashlib
import os
import tempfile
import urllib.request
from urllib.parse import quote

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def get_checkpoint(key):
    spec = MODELS[key]
    local = LOCAL_CHECKPOINTS.get(key)
    target = (Path(local).expanduser() if local else CHECKPOINT_DIR / spec['filename']).resolve()
    if local and not target.is_file():
        raise FileNotFoundError(target)
    if not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        url = ('https://huggingface.co/Mourad-zaied/SaluNet/resolve/main/'
               + quote(spec['filename']) + '?download=true')
        fd, temporary = tempfile.mkstemp(dir=target.parent, prefix='.download-', suffix='.part')
        os.close(fd)
        try:
            print('Downloading:', spec['filename'], flush=True)
            request = urllib.request.Request(url, headers={'User-Agent': 'SaluNet-CIFAR-notebook'})
            with urllib.request.urlopen(request, timeout=120) as response, open(temporary, 'wb') as out:
                total = 0
                for block in iter(lambda: response.read(1024 * 1024), b''):
                    out.write(block)
                    total += len(block)
                if total == 0:
                    raise RuntimeError('Empty download.')
                expected = response.headers.get('Content-Length')
                if expected and total != int(expected):
                    raise RuntimeError(f'Incomplete download: {total} / {expected} bytes')
            os.replace(temporary, target)
        finally:
            Path(temporary).unlink(missing_ok=True)
    if not target.is_file() or target.stat().st_size == 0:
        raise RuntimeError(f'Invalid checkpoint file: {target}')
    digest = sha256_file(target)
    print(f'{key}: {target.stat().st_size / 1024**2:.1f} MiB; SHA-256 {digest}')
    return target, digest

CHECKPOINTS = {key: get_checkpoint(key) for key in SELECTED}


## 4. FP32 evaluation

Each evaluation loads the matching architecture, evaluates all 10,000 test
images, and exits the process. The script prints its metrics at the end;
waiting without a progress bar is normal.

Complete logs, execution settings, and results are saved in `RESULTS_DIR`.
The checkpoint hash identifies the exact file evaluated; it is not a check
against an official reference hash.


In [ ]:
import re
import json
import csv
import time
from datetime import datetime, timezone
from IPython.display import display, Markdown

def parse_metrics(output):
    fields = {
        'samples': r'^Samples\s*:\s*(\d+)\s*$',
        'loss': r'^Validation loss\s*:\s*([0-9.eE+-]+)\s*$',
        'top1': r'^Validation top-1\s*:\s*([0-9.]+)%\s*$',
        'top5': r'^Validation top-5\s*:\s*([0-9.]+)%\s*$',
    }
    result = {}
    for name, pattern in fields.items():
        matches = re.findall(pattern, output, re.MULTILINE)
        if len(matches) != 1:
            raise RuntimeError(f'Expected one {name} metric, found {len(matches)}. Inspect the log.')
        result[name] = int(matches[0]) if name == 'samples' else float(matches[0])
    if result['samples'] != 10000:
        raise RuntimeError(f"Incomplete evaluation: {result['samples']} samples, expected 10000.")
    if not (0 <= result['top1'] <= result['top5'] <= 100):
        raise RuntimeError('Invalid accuracy metrics.')
    return result

RESULTS_DIR = RESULTS_DIR.expanduser().resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')

def evaluate_checkpoint(key, suffix=''):
    spec = MODELS[key]
    path, digest = CHECKPOINTS[key]
    command = [sys.executable, '-u', str(REPO_DIR / 'scripts/eval.py'),
               '--config', spec['config'], '--checkpoint', str(path),
               '--normalization', 'salu', '--activation', spec['activation'],
               '--weights', 'selected', '--batch-size', str(BATCH_SIZE),
               '--num-workers', str(NUM_WORKERS), '--device', DEVICE, '--precision', 'fp32']
    log_path = RESULTS_DIR / f'{RUN_ID}_{key}{suffix}.log'
    print(f'\nEvaluating {key} on {DEVICE} — please wait...', flush=True)
    started = time.perf_counter()
    # Stream output to the notebook and persist it even if evaluation fails.
    lines = []
    with log_path.open('w', encoding='utf-8') as log:
        with subprocess.Popen(command, cwd=REPO_DIR, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                    lines.append(line)
                code = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
    if code:
        raise RuntimeError(f'Evaluation failed (exit {code}). Log: {log_path}')
    metrics = parse_metrics(''.join(lines))
    metrics.update({
        'id': key, 'model': spec['model'], 'dataset': spec['dataset'],
        'activation': spec['activation'], 'weights': 'EMA', 'precision': 'fp32',
        'checkpoint': str(path), 'sha256': digest, 'log': str(log_path),
        'expected_top1': spec['expected_top1'], 'expected_top5': spec['expected_top5'],
        'seconds': round(time.perf_counter() - started, 2),
    })
    metrics['delta_top1_pp'] = round(metrics['top1'] - spec['expected_top1'], 4)
    metrics['delta_top5_pp'] = round(metrics['top5'] - spec['expected_top5'], 4)
    metrics['matches_reference'] = (abs(metrics['delta_top1_pp']) <= TOLERANCE_PP
                                    and abs(metrics['delta_top5_pp']) <= TOLERANCE_PP)
    return metrics

results = []
manifest_path = RESULTS_DIR / f'{RUN_ID}_results.json'
for key in SELECTED:
    results.append(evaluate_checkpoint(key))
    manifest = {
        'run_id': RUN_ID, 'repository': str(REPO_DIR), 'commit': REPO_REVISION,
        'repository_has_changes': REPO_DIRTY, 'versions': VERSIONS,
        'device': DEVICE, 'batch_size_argument': BATCH_SIZE, 'num_workers': NUM_WORKERS,
        'tolerance_pp': TOLERANCE_PP, 'results': results,
    }
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

csv_path = RESULTS_DIR / f'{RUN_ID}_results.csv'
with csv_path.open('w', newline='', encoding='utf-8') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(results[0]))
    writer.writeheader()
    writer.writerows(results)

header = '| Model | Dataset | Activation | Images | Top-1 (%) | Top-5 (%) | Top-1 difference (pp) | Reference |\n'
header += '|---|---|---|---:|---:|---:|---:|---|\n'
for row in results:
    verdict = 'Matches' if row['matches_reference'] else 'Check difference'
    header += (f"| {row['model']} | {row['dataset']} | {row['activation'].upper()} | "
               f"{row['samples']} | {row['top1']:.2f} | {row['top5']:.2f} | "
               f"{row['delta_top1_pp']:+.2f} | {verdict} |\n")
display(Markdown(header))
print('JSON:', manifest_path)
print('CSV:', csv_path)


## 5. Optional: repeat the first model's evaluation

This cell is disabled by default. It reruns a complete evaluation of the
same file in a fresh process and compares the displayed metrics.
This repeats inference; **it is not an independent training run**.


In [ ]:
REPEAT_FIRST = False
if REPEAT_FIRST:
    key = SELECTED[0]
    repeated = evaluate_checkpoint(key, suffix='_repeat')
    previous = next(row for row in results if row['id'] == key)
    comparison = {name: (previous[name], repeated[name]) for name in ('loss', 'top1', 'top5')}
    print('First evaluation / repeated evaluation:', comparison)
    print('Same displayed accuracies:',
          all(previous[name] == repeated[name] for name in ('top1', 'top5')))


## Notes

- If GPU memory is insufficient, reduce `BATCH_SIZE` and rerun evaluation.
  Small numerical differences may occur across hardware and software versions;
  the comparison tolerance is only a practical reference.
- YAML configurations define each experiment's evaluation transforms.
  The notebook preserves these pipelines instead of reusing another model's
  preprocessing. In some recipes, `val_batch_size` in the YAML controls the
  evaluation batch size.
- Stored training scores are never substituted for measurements on test
  images. Reported metrics come exclusively from the evaluation script.
- To evaluate your own checkpoint, use its matching configuration and
  architecture. The table's reference scores apply only to the released files.


## 6. Download results

This cell creates and downloads a ZIP archive containing evaluation logs,
the CSV results table, and JSON reproducibility information.
The archive does not include checkpoints or datasets.


In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive('/content/SaluNet_CIFAR_results', 'zip', RESULTS_DIR)
print('Results archive:', archive)
files.download(archive)
